# fivefold_cv.ipynb — 5-fold CV of the finalist recipes

Final validation for the fusion-pilot champion (**tower n4pull300 + userft-MRR head**)
and its 150-epoch sibling: the 258 article games are re-partitioned into 5 folds
(each game tested exactly once) instead of the fixed 77-game test split that ~47
configs were tuned on. Per fold: retrain the tower (seed 0), then 10 head seeds,
full metrics (Hit@1/Hit@5 x neutral/noname + anchor-ridge tag micro-F1).

**Multi-GPU**: one worker process per visible GPU (no distributed) pulling
(recipe x fold) jobs from a queue. **Corpus is staged into `/dev/shm` (RAM)** once
and shared by all workers.

**Upload once to `/workspace/fusion_cache/`** (~13 GB total, from the local
scratchpad `fusion_cache/`): `games.npz`, `articles.npz`, `ss_queries.npz`,
`tag_labels.npz`, `wscan_gal.npz`, `wscan_val.npz`, `wscan_pool.npy`.

Runtime: one job = tower 300 ep (~11 min on an RTX 3080, faster on A100) + 10
heads (~15 min); 10 jobs on 4 GPUs come in around 1.5 h. Resume: finished jobs
are skipped; an interrupted job restarts from its cached tower if it got that
far. Smoke test: set `RECIPES = ["n4pull150"]`, `SEEDS = 1`.

Results: `OUT_DIR/fivefold_<recipe>_fold<k>.json` + `fivefold_summary.json`.
Worker source: [`Pod/fivefold_worker.py`](fivefold_worker.py).

In [ ]:
# fivefold_cv.ipynb
# 5-fold cross-validation of the finalist recipes (tower + userft-MRR heads).
# All pod-specific paths/constants live in THIS cell -- edit here, nowhere else.
import os, subprocess

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"

# fusion_cache corpus (upload once to /workspace; file list in the data cell below)
DATA_SRC = "/workspace/fusion_cache"
# RAM staging target -- the corpus is copied here once so every GPU worker reads
# at RAM speed (pods have plenty of RAM; ~13 GB needed).
DATA_RAM = "/dev/shm/fusion_cache"
OUT_DIR = "/workspace/fivefold_out"     # results + logs + per-fold tower caches

RECIPES = ["ice_clean", "ce_clean", "byol_clean"]  # DECON factorization (review-only towers). Prior: v4art_ft020/ultimate/byol/q1pull300/n4pull300/n4fresh300 (done cells auto-skip)
N_FOLDS = 5
SEEDS = 10
HEAD_EPOCHS = 600                        # phase-2 cap; val-MRR early-stops ~ep100

def _detect_gpus():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=5).stdout.strip()
        ids = [l.strip() for l in out.splitlines() if l.strip()]
        return ids if ids else ["0"]
    except Exception:
        return ["0"]

GPUS = _detect_gpus()
os.makedirs(OUT_DIR, exist_ok=True)
print("repo :", REPO)
print("data :", DATA_SRC, "->", DATA_RAM, "(RAM staging)")
print("out  :", OUT_DIR)
print("jobs :", [f"{r}/fold{k}" for r in RECIPES for k in range(N_FOLDS)])
print("gpus :", GPUS, "(one worker per GPU)")

In [ ]:
# Clone or FORCE-sync to origin/main before every run. The pod repo is a MIRROR
# of GitHub: reset --hard discards pod-local edits in tracked paths (push to
# GitHub instead). Untracked files (corpus, results, logs) are untouched.
import os, importlib.util

if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}

%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD

for pkg in ("sklearn", "scipy", "h5py"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy h5py
        break

In [ ]:
# Verify the corpus and stage it into RAM (/dev/shm). One copy is shared by all
# GPU workers -- they mmap it, so per-worker extra RAM stays near zero.
import shutil
from pathlib import Path

REQUIRED = ["games.npz", "articles.npz", "ss_queries.npz", "tag_labels.npz",
            "wscan_gal.npz", "wscan_val.npz", "wscan_pool.npy"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing} -- upload fusion_cache first"

dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
total = sum((dst / f).stat().st_size for f in REQUIRED) / 1e9
print(f"corpus in RAM: {DATA_DIR} ({total:.1f} GB)")

In [ ]:
# Run all (recipe x fold) jobs across the GPUs on THIS machine: one subprocess
# per GPU, pinned via CUDA_VISIBLE_DEVICES, pulling jobs from a shared queue.
# Resume by default -- a job whose result JSON exists is skipped (delete the JSON
# to redo it; delete its tower_*.npz too for a full retrain). Per-job logs land
# in OUT_DIR/logs/. Re-run this cell after any interruption.
import os, queue, subprocess, threading, time
from pathlib import Path

jobs = queue.Queue()
n_jobs = 0
for r in RECIPES:
    for k in range(N_FOLDS):
        if (Path(OUT_DIR) / f"fivefold_{r}_fold{k}.json").exists():
            print(f"[skip] {r}/fold{k} already done")
            continue
        jobs.put((r, k)); n_jobs += 1
log_dir = Path(OUT_DIR) / "logs"
log_dir.mkdir(exist_ok=True)
fails = []

def worker(gpu):
    while True:
        try:
            r, k = jobs.get_nowait()
        except queue.Empty:
            return
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu)
        log = log_dir / f"{r}_fold{k}.log"
        print(f"[gpu{gpu}] start {r}/fold{k} -> logs/{log.name}", flush=True)
        t0 = time.time()
        with open(log, "w") as fh:
            p = subprocess.run(
                ["python", "-u", os.path.join(REPO, "Pod/fivefold_worker.py"),
                 "--data-dir", DATA_DIR, "--out-dir", OUT_DIR, "--repo", REPO,
                 "--recipe", r, "--fold", str(k),
                 "--n-folds", str(N_FOLDS), "--seeds", str(SEEDS),
                 "--head-epochs", str(HEAD_EPOCHS)],
                stdout=fh, stderr=subprocess.STDOUT, env=env)
        status = "ok" if p.returncode == 0 else f"FAIL rc={p.returncode}"
        if p.returncode != 0:
            fails.append((r, k, str(log)))
        print(f"[gpu{gpu}] {status} {r}/fold{k} [{(time.time()-t0)/60:.1f} min]", flush=True)

threads = [threading.Thread(target=worker, args=(g,)) for g in GPUS]
t0 = time.time()
for t in threads: t.start()
for t in threads: t.join()
print(f"\nfinished in {(time.time()-t0)/60:.1f} min; {n_jobs} run, {len(fails)} failed")
for r, k, log in fails:
    print("  FAILED:", r, f"fold{k}", "-> check", log)

In [ ]:
# Aggregate: per-fold table, overall mean over all runs (folds x seeds),
# between-fold std of fold means, and the paired recipe comparison per fold.
import json
import numpy as np
from pathlib import Path

KEYS = ["h1_neutral", "h5_neutral", "h1_noname", "h5_noname", "tag_neutral", "tag_noname"]
summary = {}
for r in RECIPES:
    per_fold = []
    for k in range(N_FOLDS):
        p = Path(OUT_DIR) / f"fivefold_{r}_fold{k}.json"
        if not p.exists():
            print(f"[warn] {r}: missing {p.name} -- finish the run cell first")
            break
        per_fold.append(json.loads(p.read_text()))
    if len(per_fold) < N_FOLDS:
        continue
    print(f"\n=== {r} (per-fold mean+-seed-std, {SEEDS} seeds) ===")
    print("fold |  n | " + " | ".join(f"{k:12s}" for k in KEYS))
    fold_means = {k: [] for k in KEYS}
    all_runs = {k: [] for k in KEYS}
    for fr in per_fold:
        row = []
        for k in KEYS:
            vals = [s[k] for s in fr["seeds"]]
            fold_means[k].append(float(np.mean(vals)))
            all_runs[k] += vals
            row.append(f"{np.mean(vals):.3f}+-{np.std(vals):.3f}")
        print(f"  {fr['fold']}  | {len(fr['test_games']):2d} | " + " | ".join(row))
    print("overall  (all runs)           | " + " | ".join(
        f"{np.mean(all_runs[k]):.3f}+-{np.std(all_runs[k]):.3f}" for k in KEYS))
    print("fold means (between-fold std) | " + " | ".join(
        f"{np.mean(fold_means[k]):.3f}+-{np.std(fold_means[k]):.3f}" for k in KEYS))
    summary[r] = {k: {"overall_mean": float(np.mean(all_runs[k])),
                      "overall_std": float(np.std(all_runs[k])),
                      "fold_means": fold_means[k],
                      "between_fold_std": float(np.std(fold_means[k]))} for k in KEYS}

done = [r for r in RECIPES if r in summary]
for i in range(len(done)):
    for j in range(i + 1, len(done)):
        a, b = done[i], done[j]
        print(f"\n=== paired fold-mean diff: {a} - {b} ===")
        for k in KEYS:
            d = np.array(summary[a][k]["fold_means"]) - np.array(summary[b][k]["fold_means"])
            print(f"{k:12s}: {d.mean():+.3f}+-{d.std():.3f}   {a} wins {(d>0).sum()}/{len(d)} folds")

out = Path(OUT_DIR) / "fivefold_summary.json"
out.write_text(json.dumps(summary, indent=2))
print("\nsaved", out)